# QM640 · 02 Public Jira Dataset subset (Colab)
Downloads the 5.8 GB Zenodo archive to Colab's **local** disk (not Drive), streams the Apache, Red Hat and MongoDB collections straight out of the MongoDB dump, keeps resolved issues with the fields we need, and saves compact Parquet files plus a summary to Drive.

**Runtime:** CPU, *High-RAM* if available (Runtime → Change runtime type). About 30–60 minutes. Then *Runtime → Run all*.

In [ ]:
# 1 · Drive and folders
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys, os
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
RESULTS, OUT_DRIVE = DATA_ROOT/'results', DATA_ROOT/'processed'/'jira'
LOCAL = Path('/content/jira'); (LOCAL/'out').mkdir(parents=True, exist_ok=True)
for p in (RESULTS, OUT_DRIVE): p.mkdir(parents=True, exist_ok=True)
print('free local disk (GB):', round(shutil.disk_usage('/content').free / 1e9, 1))

In [ ]:
# 2 · Libraries
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymongo', 'pyarrow'], check=True)

In [ ]:
%%writefile /content/jira_lib.py
"""Stream the Public Jira Dataset MongoDB dump straight out of the zip and
flatten each issue into compact columns. No MongoDB server is needed when the
dump is in directory format (<collection>.bson or .bson.gz)."""
import gzip, io, json, re, zipfile
import pyarrow as pa, pyarrow.parquet as pq
import bson

STR = pa.string(); I = pa.int64()
SCHEMA = pa.schema([(c, STR) for c in ['repository','key','project','issuetype','priority','status','resolution','created','resolutiondate','summary','description','components','labels','assignee']]
                   + [('n_comments', I), ('last_comment', STR), ('issuelinks', STR), ('n_links', I), ('has_duplicate_link', pa.bool_()),
                      ('n_assignee_changes', I), ('n_component_changes', I), ('n_priority_changes', I), ('n_type_changes', I), ('n_history_items', I)])

TARGET_REPOS = ["Apache", "RedHat", "MongoDB"]

def g(d, path):
    for p in path.split('.'):
        if not isinstance(d, dict): return None
        d = d.get(p)
    return d

def name_of(x):
    if isinstance(x, dict): return x.get('name') or x.get('value') or x.get('key')
    return x

def trunc(s, n):
    if not isinstance(s, str): return None
    s = re.sub(r'\{code[^}]*\}.*?\{code\}', ' [code] ', s, flags=re.S)   # shorten code blocks
    s = re.sub(r'\s+', ' ', s).strip()
    return s[:n]

def flatten(doc, repo):
    f = doc.get('fields') or {}
    links = []
    for ln in f.get('issuelinks') or []:
        t = (ln.get('type') or {})
        if 'outwardIssue' in ln: links.append({'type': t.get('name'), 'dir': 'out', 'label': t.get('outward'), 'key': (ln['outwardIssue'] or {}).get('key')})
        elif 'inwardIssue' in ln: links.append({'type': t.get('name'), 'dir': 'in', 'label': t.get('inward'), 'key': (ln['inwardIssue'] or {}).get('key')})
    comments = (f.get('comment') or {}).get('comments') or []
    hist = (doc.get('changelog') or {}).get('histories') or []
    changes = {}
    for h in hist:
        for it in h.get('items') or []:
            fld = (it.get('field') or '').lower()
            changes[fld] = changes.get(fld, 0) + 1
    assignee = f.get('assignee')
    return {
        'repository': repo,
        'key': doc.get('key'),
        'project': name_of(g(f, 'project.key')) or name_of(f.get('project')),
        'issuetype': name_of(f.get('issuetype')),
        'priority': name_of(f.get('priority')),
        'status': name_of(f.get('status')),
        'resolution': name_of(f.get('resolution')),
        'created': str(f.get('created')) if f.get('created') else None,
        'resolutiondate': str(f.get('resolutiondate')) if f.get('resolutiondate') else None,
        'summary': trunc(f.get('summary'), 300),
        'description': trunc(f.get('description'), 2000),
        'components': json.dumps([name_of(c) for c in f.get('components') or []]),
        'labels': json.dumps(list(f.get('labels') or [])),
        'assignee': str((assignee.get('key') or assignee.get('name') or assignee.get('accountId')) if isinstance(assignee, dict) else assignee) if assignee else None,
        'n_comments': len(comments),
        'last_comment': trunc(comments[-1].get('body'), 800) if comments else None,
        'issuelinks': json.dumps(links),
        'n_links': len(links),
        'has_duplicate_link': any((l['type'] or '').lower() == 'duplicate' for l in links),
        'n_assignee_changes': changes.get('assignee', 0),
        'n_component_changes': changes.get('component', 0),
        'n_priority_changes': changes.get('priority', 0),
        'n_type_changes': changes.get('issuetype', 0),
        'n_history_items': sum(changes.values()),
    }

def dump_members(names, repos=TARGET_REPOS):
    """Map repo -> zip member holding its BSON collection."""
    out = {}
    for r in repos:
        pat = re.compile(rf'(^|/){re.escape(r)}\.bson(\.gz)?$', re.I)
        m = [n for n in names if pat.search(n)]
        if m: out[r] = m[0]
    return out

def stream_repo(zf, member, repo, writer_path, only_resolved=True, batch=20000, log=print):
    raw = zf.open(member)
    fh = gzip.GzipFile(fileobj=raw) if member.endswith('.gz') else raw
    fh = io.BufferedReader(fh, buffer_size=1 << 20)
    rows, n_seen, n_kept, writer = [], 0, 0, None
    for doc in bson.decode_file_iter(fh):
        n_seen += 1
        if only_resolved and not g(doc, 'fields.resolution'):
            continue
        rows.append(flatten(doc, repo)); n_kept += 1
        if len(rows) >= batch:
            t = pa.Table.from_pylist(rows, schema=SCHEMA); writer = writer or pq.ParquetWriter(writer_path, SCHEMA); writer.write_table(t); rows = []
            if n_kept % 100000 < batch: log(f'  {repo}: seen {n_seen:,}, kept {n_kept:,}')
    if rows:
        t = pa.Table.from_pylist(rows, schema=SCHEMA); writer = writer or pq.ParquetWriter(writer_path, SCHEMA); writer.write_table(t)
    if writer: writer.close()
    return n_seen, n_kept


# ---------------------------------------------------------------------------
# mongodump --archive format reader (single-file dump, optionally gzipped)
# Layout: magic 0x8199e26d | prelude header doc | collection-metadata docs | -1
#         then blocks: namespace header {db, collection, EOF, CRC} | docs... | -1
# ---------------------------------------------------------------------------
import struct
MAGIC = 0x8199e26d
TERM = 0xFFFFFFFF

def _open_archive(zf, member):
    raw = zf.open(member)
    head = raw.read(2); raw.close(); raw = zf.open(member)
    fh = gzip.GzipFile(fileobj=raw) if head == b'\x1f\x8b' else raw
    return io.BufferedReader(fh, buffer_size=4 << 20)

def _read_doc(fh):
    """Return (raw_bytes or None for terminator); raises EOFError at end."""
    b = fh.read(4)
    if len(b) < 4: raise EOFError
    n = struct.unpack('<I', b)[0]
    if n == TERM: return None
    rest = fh.read(n - 4)
    if len(rest) < n - 4: raise EOFError
    return b + rest

def archive_info(zf, member):
    """Read only the prelude: tool/server versions and the list of collections."""
    fh = _open_archive(zf, member)
    magic = struct.unpack('<I', fh.read(4))[0]
    if magic != MAGIC: raise ValueError(f'not a mongodump archive (magic {magic:#x})')
    header = bson.decode(_read_doc(fh))
    cols = []
    while True:
        d = _read_doc(fh)
        if d is None: break
        m = bson.decode(d); cols.append({'db': m.get('db'), 'collection': m.get('collection'), 'size': m.get('size')})
    return header, cols

def stream_archive(zf, member, repos, out_dir, only_resolved=True, batch=20000, log=print):
    """One pass over the archive; flatten documents of the wanted collections."""
    fh = _open_archive(zf, member)
    assert struct.unpack('<I', fh.read(4))[0] == MAGIC
    _read_doc(fh)                                   # prelude header
    while _read_doc(fh) is not None: pass           # collection metadata
    wanted = {r.lower(): r for r in repos}
    writers, rows, seen, kept = {}, {r: [] for r in repos}, {r: 0 for r in repos}, {r: 0 for r in repos}
    current, expect_header, n_blocks = None, True, 0
    def flush(r):
        if rows[r]:
            t = pa.Table.from_pylist(rows[r], schema=SCHEMA)
            if r not in writers: writers[r] = pq.ParquetWriter(f'{out_dir}/{r}.parquet', SCHEMA)
            writers[r].write_table(t); rows[r] = []
    try:
        while True:
            d = _read_doc(fh)
            if d is None: expect_header = True; continue
            if expect_header:
                h = bson.decode(d); n_blocks += 1
                current = wanted.get(str(h.get('collection', '')).lower()); expect_header = False
                continue
            if current is None: continue            # skip documents of other collections cheaply
            doc = bson.decode(d); seen[current] += 1
            if only_resolved and not g(doc, 'fields.resolution'): continue
            rows[current].append(flatten(doc, current)); kept[current] += 1
            if len(rows[current]) >= batch:
                flush(current)
                if kept[current] % 100000 < batch: log(f'  {current}: seen {seen[current]:,}, kept {kept[current]:,}')
    except EOFError:
        pass
    for r in repos: flush(r)
    for w in writers.values(): w.close()
    return {r: {'issues_in_dump': seen[r], 'resolved_kept': kept[r]} for r in repos}, n_blocks


In [ ]:
# 3 · Download (resumable) and verify the checksum
URL = 'https://zenodo.org/records/15719919/files/2025-06-23%20ThePublicJiraDataset.zip?download=1'
ZIP = LOCAL/'ThePublicJiraDataset.zip'
subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', str(ZIP), URL], check=True)
md5 = subprocess.run(['md5sum', str(ZIP)], capture_output=True, text=True).stdout.split()[0]
print('size (GB):', round(ZIP.stat().st_size / 1e9, 2), '| md5:', md5, '| OK' if md5 == '02f85309d966092ea130ca0797aea795' else '| MISMATCH - re-run this cell')

In [ ]:
# 4 · Find the dump inside the zip and read its table of contents (saved to Drive first)
import zipfile, sys, json
sys.path.insert(0, '/content'); import importlib, jira_lib as J; importlib.reload(J)
zf = zipfile.ZipFile(ZIP)
names = [i.filename for i in zf.infolist()]
(RESULTS/'step02_zip_listing.txt').write_text('\n'.join(f'{i.file_size/1e6:10.1f} MB  {i.filename}' for i in zf.infolist()))
ARCH = next((n for n in names if n.endswith('.archive')), None)
print('archive member:', ARCH)
try:
    header, cols = J.archive_info(zf, ARCH)
except Exception as e:
    with zf.open(ARCH) as fh: first = fh.read(64).hex()
    (RESULTS/'step02_archive_info.md').write_text(f'ERROR {e}\nfirst bytes: {first}')
    raise RuntimeError('Archive format not recognised - details saved to results/step02_archive_info.md.')
REPOS = [c['collection'] for c in cols if any(k in str(c['collection']).lower() for k in ('apache', 'redhat', 'mongo'))]
(RESULTS/'step02_archive_info.md').write_text('# Archive table of contents\n\n' + json.dumps(header, default=str, indent=1) + '\n\n' + json.dumps(cols, default=str, indent=1) + f'\n\nSelected: {REPOS}')
print('collections:', [c['collection'] for c in cols]); print('selected:', REPOS)
assert REPOS, 'No Apache/RedHat/MongoDB collections found - see results/step02_archive_info.md'


In [ ]:
# 5 · One pass over the archive: keep resolved issues of the selected repositories (about 20-40 minutes)
import time
t0 = time.time()
stats, n_blocks = J.stream_archive(zf, ARCH, REPOS, str(LOCAL/'out'))
print('blocks read:', n_blocks, '| minutes:', round((time.time() - t0) / 60, 1)); print(stats)
stats = {r: s for r, s in stats.items() if s['resolved_kept'] > 0}
assert stats, 'No issues extracted'


In [ ]:
# 6 · Profile, copy to Drive, write the step summary
import pandas as pd, json, datetime
frames = {r: pd.read_parquet(LOCAL/'out'/f'{r}.parquet') for r in stats}
jira = pd.concat(frames.values(), ignore_index=True)
for c in ('created', 'resolutiondate'):
    jira[c] = pd.to_datetime(jira[c], errors='coerce', utc=True)
jira['resolution_hours'] = (jira['resolutiondate'] - jira['created']).dt.total_seconds() / 3600
jira['month'] = jira['created'].dt.to_period('M').astype(str)
by_repo = jira.groupby('repository').agg(issues=('key', 'size'), projects=('project', 'nunique'),
    first=('created', 'min'), last=('created', 'max'), duplicate_links=('has_duplicate_link', 'sum'),
    median_resolution_h=('resolution_hours', 'median'), with_components=('components', lambda s: (s != '[]').mean()))
top_proj = (jira.groupby(['repository', 'project']).agg(issues=('key', 'size'), dup=('has_duplicate_link', 'sum'),
    months=('month', 'nunique')).sort_values('issues', ascending=False).head(25))
monthly = jira.groupby(['repository', 'month']).size().unstack(0).fillna(0)
dense = (monthly >= 200).sum()
for r in stats:
    shutil.copy(LOCAL/'out'/f'{r}.parquet', OUT_DRIVE/f'{r}.parquet')
sizes = {r: round((OUT_DRIVE/f'{r}.parquet').stat().st_size / 1e6, 1) for r in stats}
def mdt(df):
    try: return df.to_markdown()
    except Exception: return '```\n' + df.to_string() + '\n```'
lines = [f'# Step 02 summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', '', '## Streaming stats', '```', json.dumps(stats, indent=1), '```',
         '## By repository', mdt(by_repo), '', '## Months with >= 200 resolved issues (RQ4 needs 25)', mdt(dense.to_frame('months')), '',
         '## Top 25 projects', mdt(top_proj), '', '## Issue types', mdt(jira['issuetype'].value_counts().head(15).to_frame()), '',
         '## Resolutions', mdt(jira['resolution'].value_counts().head(15).to_frame()), '',
         '## Priority', mdt(jira['priority'].value_counts().head(10).to_frame()), '',
         '## Parquet sizes on Drive (MB)', json.dumps(sizes), '', '## Columns', ', '.join(jira.columns)]
(RESULTS/'step02_summary.md').write_text('\n'.join(lines))
print('\n'.join(lines[:40]))

Done. The 5.8 GB zip on Colab's disk is deleted automatically when the session ends.